# Classification Trees, Random Forest and Gradient Boosting – Titanic

**Dataset:** 891 passengers of the Titanic, which sank on 15 April 1912 after hitting an iceberg (Kaggle).
File: `../00_Data/titanic.csv`. This is the data set behind the classification examples of the concept notebooks.

| variable | meaning |
|---|---|
| `Survived` | 1 = survived, 0 = died – **target** |
| `Pclass` | ticket class (1 = first, 2 = second, 3 = third) |
| `Sex`, `Age` | sex, age in years (missing for 177 passengers) |
| `SibSp`, `Parch` | number of siblings/spouses and parents/children aboard |
| `Fare` | ticket price (British pounds) |
| `Embarked` | port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton) |
| `Name`, `Ticket`, `Cabin` | not used here |

**Question:** Which passengers had the best chances of survival? How well can survival be predicted from sex, age,
class and fare?

**After this exercise you can ...**
- compute Gini impurity and the impurity decrease of a split by hand,
- fit, visualise, interpret and prune a classification tree and read class probabilities from its leaves,
- explain how a random forest averages class probabilities,
- tune random forest and gradient boosting classifiers with `GridSearchCV` and compare them with ROC/AUC.

## Concept

The concepts behind this exercise (formulas, worked examples and additional explanations) are
explained in the notebook [classification_trees_ensembles_concept.ipynb](classification_trees_ensembles_concept.ipynb). Read it before you start.

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
)
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.inspection import permutation_importance

# Ignore warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: Calculations by hand

**a) Gini impurity.** A node contains 200 passengers, 76 of whom survived. Two candidate splits are evaluated:

| candidate split | left node (yes) | right node (no) |
|---|---|---|
| `male?` | 130 passengers: 25 survived / 105 not | 70 passengers: 51 survived / 19 not |
| `1st class?` | 50 passengers: 30 survived / 20 not | 150 passengers: 46 survived / 104 not |

Compute the Gini impurity of the parent node, $G_L$, $G_R$, the weighted impurity and $\Delta G$ for both candidate
splits. Which split does the tree choose?

**b) Leaf probability.** A leaf contains 145 survivors and 125 non-survivors (example from the concept notebook). What does it predict for a
new passenger who ends up in this leaf (class and probability)?

**c) Random forest.** Five trees give the probabilities of survival 0.9, 0.7, 0.4, 0.6 and 0.3 for a passenger. What
does the forest predict? What would a classic majority vote predict?

**d)** Verify your results with Python.

In [ ]:
# Exercise 1: Gini impurity by hand


# Function: computes the Gini impurity from the class counts
def gini(n_survived, n_died):
    """Gini impurity of a node with two classes."""
    n = n_survived + n_died
    return 1 - ((n_survived / n) ** 2 + (n_died / n) ** 2)


# a) Gini impurity of the parent node and of both splits

# Parent node: 200 passengers, 76 survived / 124 not
g_parent = gini(76, 124)
print('Parent: G =', round(g_parent, 4))

# Split "male?": left 130 passengers (25 / 105), right 70 (51 / 19)
g_left = gini(25, 105)
g_right = gini(51, 19)
g_weighted = 130 / 200 * g_left + 70 / 200 * g_right
print(
    'male?      : G_L =',
    round(g_left, 4),
    ' G_R =',
    round(g_right, 4),
    ' weighted G =',
    round(g_weighted, 4),
    ' Delta G =',
    round(g_parent - g_weighted, 4),
)

# Split "1st class?": left 50 passengers (30 / 20), right 150 (46 / 104)
g_left = gini(30, 20)
g_right = gini(46, 104)
g_weighted = 50 / 200 * g_left + 150 / 200 * g_right
print(
    '1st class? : G_L =',
    round(g_left, 4),
    ' G_R =',
    round(g_right, 4),
    ' weighted G =',
    round(g_weighted, 4),
    ' Delta G =',
    round(g_parent - g_weighted, 4),
)

# b) Leaf probability
print('Leaf 145 : 125 -> P(survived) =', round(145 / 270, 3))

# c) Random forest: mean probability vs. majority vote
p_trees = np.array([0.9, 0.7, 0.4, 0.6, 0.3])
print('Mean probability =', round(p_trees.mean(), 2))
print('Predict survived:', p_trees.mean() >= 0.5)
print('Majority vote:', np.sum(p_trees >= 0.5), 'of 5 trees vote survived')

**Solution:**

a) Parent: $G = 1-(0.38^2+0.62^2) = 1-(0.1444+0.3844) \approx 0.471$.

| split | $G_L$ | $G_R$ | weighted $G$ | $\Delta G$ |
|---|---|---|---|---|
| `male?` | $1-(0.192^2+0.808^2)\approx0.311$ | $1-(0.729^2+0.271^2)\approx0.396$ | $0.65\cdot0.311+0.35\cdot0.396\approx0.340$ | **≈ 0.131** |
| `1st class?` | $1-(0.6^2+0.4^2)=0.48$ | $1-(0.307^2+0.693^2)\approx0.425$ | $0.25\cdot0.48+0.75\cdot0.425\approx0.439$ | ≈ 0.032 |

`male?` reduces the impurity much more and becomes the split – as in the example tree.

b) $\hat p(\text{survived}) = 145/270 \approx 0.537$ → predict "survived" (with $\tau = 0.5$), but with a probability only
slightly above 0.5.

c) Mean probability $= (0.9+0.7+0.4+0.6+0.3)/5 = 0.58$ → the forest predicts "survived". A majority vote also gives
"survived" (3 of 5 trees ≥ 0.5). Averaging the probabilities (scikit-learn) and voting (classic random forest) usually,
but not always, give the same class.

## Exercise 2: Import and explore the data

a) Import the data. How many passengers survived? Which variables have missing values?

b) Compute the survival rate by `Sex`, by `Pclass` and by `Sex` × `Pclass`. Plot the survival rate by age group
(0–9, 10–19, …, 70–79 years) for women and men.

c) Prepare the features: `male` (1 = male), `Age` (missing values replaced by the median age), `Pclass`, `SibSp`,
`Parch`, `Fare` and dummy variables for `Embarked` (missing values replaced by `'S'`). Split the data into a stratified
training (80 %) and test set (20 %) with `random_state=42`.

In [ ]:
# a) Import the data as titanic; survivors and missing values
titanic = pd.read_csv('../00_Data/titanic.csv', sep=',')
print(titanic.shape)
print(
    'Survived:',
    titanic['Survived'].sum(),
    ' share:',
    round(titanic['Survived'].mean(), 3),
)
print(titanic.isna().sum())

# b) Survival rates and figure by age group

# Survival rate by sex, by class and by sex and class
print(titanic.groupby('Sex')['Survived'].mean().round(3))
print(titanic.groupby('Pclass')['Survived'].mean().round(3))
print(titanic.groupby(['Sex', 'Pclass'])['Survived'].mean().round(3))

# Age groups of 10 years
titanic['age_group'] = pd.cut(
    titanic['Age'],
    bins=[0, 10, 20, 30, 40, 50, 60, 70, 80],
    right=False,
    labels=[
        '0-9',
        '10-19',
        '20-29',
        '30-39',
        '40-49',
        '50-59',
        '60-69',
        '70-79',
    ],
)

# Figure showing the survival rate by age group for women and men
rate = titanic.pivot_table(
    index='age_group', columns='Sex', values='Survived', observed=True
)
rate.plot(kind='bar')
plt.ylabel('Survival rate')
plt.xlabel('Age group [years]')
plt.show()

# c) Features: male (1 = male), Age (missing -> median), Pclass, SibSp,
#    Parch, Fare and Embarked (missing -> 'S', as dummy variables)
titanic['male'] = (titanic['Sex'] == 'male').astype(int)
titanic['Age'] = titanic['Age'].fillna(titanic['Age'].median())
titanic['Embarked'] = titanic['Embarked'].fillna('S')
features = ['male', 'Age', 'Pclass', 'SibSp', 'Parch', 'Fare', 'Embarked']
X = pd.get_dummies(
    titanic[features], columns=['Embarked'], drop_first=True, dtype=int
)
y = titanic['Survived']

# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print('Training set:', X_train.shape, ' Test set:', X_test.shape)

**Solution:**

a) 342 of 891 passengers survived (38.4 %). `Age` is missing for 177 passengers, `Cabin` for 687 (not used),
`Embarked` for 2.

b) Women survived far more often (74 %) than men (19 %), first-class passengers (63 %) more often than second (47 %) and
third class (24 %). The combination is striking: almost all women in 1st and 2nd class survived (97 % and 92 %), but only
half of the women in 3rd class; men in 2nd and 3rd class had the lowest chances (16 % and 14 %). Among the men, boys
under 10 years had a much higher survival rate than adult men – "women and children first".

c) Replacing missing ages by the median is a simple solution; strictly speaking, the median should be computed on the
training data only.

## Exercise 3: The example tree

Section «Anatomy of a classification tree» of the concept notebook shows a tree with the questions *male?*, *age ≤ 9.5?*, *3rd class?* and
*fare ≤ 23?*.

a) Fit a `DecisionTreeClassifier(max_depth=3, random_state=42)` on **all 891 passengers** with the features `male`,
`Age`, `third_class` (1 = 3rd class) and `Fare`, and plot it with `tree.plot_tree(..., class_names=['died', 'survived'])`.

b) Compare the tree with the example tree: which questions and thresholds are the same? Determine depth (`get_depth()`), size
(`tree_.node_count`) and number of leaves.

c) Check the root split by hand: compute the Gini impurity of the root and of its two children from the class counts
printed below, and $\Delta G$.

d) Use the tree to predict the survival probability of (1) a 30-year-old woman in 3rd class with a fare of 10 pounds and
(2) a 5-year-old boy in 2nd class with a fare of 30 pounds. Which leaves do they end up in?

In [ ]:
# a) Features of the example tree (all 891 passengers)
X_example = pd.DataFrame(
    {
        'male': X['male'],
        'Age': X['Age'],
        'third_class': (X['Pclass'] == 3).astype(int),
        'Fare': X['Fare'],
    }
)

# Fit the tree
example_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
example_tree.fit(X_example, y)

# Figure showing the fitted tree
plt.figure(figsize=(20, 7))
tree.plot_tree(
    example_tree,
    feature_names=list(X_example.columns),
    class_names=['died', 'survived'],
    filled=True,
)
plt.show()

# The same tree as text
print(tree.export_text(example_tree, feature_names=list(X_example.columns)))

# b) Depth, size and number of leaves
print('Depth:', example_tree.get_depth())
print('Size (nodes):', example_tree.tree_.node_count)
print('Leaves:', example_tree.get_n_leaves())

# c) Class counts of the root (all passengers) and of its two children
#    (the root asks "male?": women = 0, men = 1)
print(y.value_counts())
print(pd.crosstab(X_example['male'], y))

# Gini of the root and of the children, Delta G (use gini() from Exercise 1)
g_root = gini(342, 549)
g_women = gini(233, 81)
g_men = gini(109, 468)
g_weighted = 314 / 891 * g_women + 577 / 891 * g_men
print(
    'Root: G =',
    round(g_root, 3),
    ' women: G =',
    round(g_women, 3),
    ' men: G =',
    round(g_men, 3),
)
print('Delta G of the root split =', round(g_root - g_weighted, 3))

# d) Two new passengers
new_passengers = pd.DataFrame(
    {'male': [0, 1], 'Age': [30, 5], 'third_class': [1, 0], 'Fare': [10, 30]}
)

# Survival probabilities and leaf numbers (example_tree.apply)
# (columns of predict_proba: died, survived)
print(example_tree.predict_proba(new_passengers).round(3))
print('Leaf ids:', example_tree.apply(new_passengers))

**Solution:**

b) The tree is very similar to the example tree: the root asks **male?**; for men the next question is the **age**
(≤ 6.5 years instead of ≤ 9.5 in the example tree); for women it asks **3rd class?** and then **fare ≤ 23.35** (example tree: 23).
The thresholds differ slightly because the example tree presumably used a different handling of the missing ages. Depth 3,
size 15 nodes (the maximum for depth 3 – the example tree was pruned to 9 nodes), 8 leaves.

c) Root: $G = 1-\big((549/891)^2+(342/891)^2\big) \approx 0.473$; women (314 passengers, 233 survived):
$G \approx 0.383$; men (577, 109 survived): $G \approx 0.306$.
$\Delta G = 0.473 - (314/891\cdot 0.383 + 577/891\cdot 0.306) \approx 0.140$ – identical to the printed values.

d) (1) The 30-year-old woman in 3rd class with a cheap ticket ends up in the leaf "female, 3rd class, fare ≤ 23.35" with
69 survivors and 48 non-survivors → $\hat p(\text{survived}) = 69/117 \approx 0.59$. (2) The 5-year-old boy in 2nd class
ends up in the leaf "male, age ≤ 6.5, not 3rd class" with 10 survivors out of 10 → $\hat p = 1.0$ (all boys in 1st and
2nd class survived).

## Exercise 4: Overfitting and pruning

From now on we use all prepared features and the training/test split of Exercise 2.

a) Fit a tree with `max_depth=3` and a tree without depth limit on the training set. Compare training accuracy, test
accuracy and test AUC. How many leaves does the full tree have?

b) Choose `ccp_alpha` with `GridSearchCV` (stratified 5-fold CV, `scoring='roc_auc'`) from the values of
`cost_complexity_pruning_path(X_train, y_train).ccp_alphas`. How many leaves does the best tree have, and what is its test
AUC?

In [ ]:
# Exercise 4: overfitting and pruning
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# a) Depth-3 tree (tree_3) and full tree (full_tree): leaves, accuracy, AUC
tree_3 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_3.fit(X_train, y_train)
full_tree = DecisionTreeClassifier(random_state=42)
full_tree.fit(X_train, y_train)

# Leaves, accuracy and AUC of both trees
for model in [tree_3, full_tree]:
    p_test = model.predict_proba(X_test)[:, 1]
    print(
        'Leaves:',
        model.get_n_leaves(),
        ' train accuracy:',
        round(model.score(X_train, y_train), 3),
        ' test accuracy:',
        round(model.score(X_test, y_test), 3),
        ' test AUC:',
        round(roc_auc_score(y_test, p_test), 3),
    )

# b) Choose ccp_alpha with GridSearchCV (store the best tree as pruned_tree)
alphas = full_tree.cost_complexity_pruning_path(X_train, y_train).ccp_alphas
grid_tree = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    {'ccp_alpha': alphas},
    cv=cv,
    scoring='roc_auc',
)
grid_tree.fit(X_train, y_train)
pruned_tree = grid_tree.best_estimator_
print('Best ccp_alpha:', round(grid_tree.best_params_['ccp_alpha'], 4))
print('CV AUC:', round(grid_tree.best_score_, 3))
print('Leaves of the pruned tree:', pruned_tree.get_n_leaves())
p_test = pruned_tree.predict_proba(X_test)[:, 1]
print('Test AUC of the pruned tree:', round(roc_auc_score(y_test, p_test), 3))

**Solution:**

a) The full tree has 153 leaves and classifies the training data almost perfectly (training accuracy 0.98) – classic
overfitting. On the test set its accuracy (0.82) is even slightly higher than that of the depth-3 tree (0.79), but its
AUC is lower (0.80 vs. 0.82): almost all leaves of the full tree are pure, so its probabilities are mostly 0 or 1 and the
ranking information is lost.

b) Pruning selects a tree with 17 leaves (CV AUC ≈ 0.84). On the test set its AUC (≈ 0.78) is lower than that of the
depth-3 tree. With only 179 test passengers, differences of a few hundredths are **not meaningful** – the CV estimate
(averaged over 5 folds) is more reliable than one small test set.

## Exercise 5: Look inside a random forest

a) Fit a `RandomForestClassifier(n_estimators=500, min_samples_leaf=5, oob_score=True, random_state=42)` on the
training set (`min_samples_leaf=5`: each leaf contains at least 5 passengers, so every tree outputs class shares) and print the
OOB accuracy (`oob_score_`) and the test accuracy.

b) Take the first passenger of the test set. Print the survival probabilities of the first 10 trees
(`rf.estimators_[i].predict_proba(...)`) and the mean over **all** 500 trees. Compare the mean with
`rf.predict_proba()`.

In [ ]:
# a) Random forest with OOB score
rf = RandomForestClassifier(
    n_estimators=500,
    min_samples_leaf=5,
    oob_score=True,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
print('OOB accuracy:', round(rf.oob_score_, 3))
print('Test accuracy:', round(rf.score(X_test, y_test), 3))

# b) Probabilities of single trees for one passenger

# First passenger of the test set
passenger = X_test.iloc[[0]]
print(passenger)

# Survival probability of each of the 500 trees (rf.estimators_)
# (.values: the single trees were fitted without column names)
p_trees = []
for single_tree in rf.estimators_:
    p_trees.append(single_tree.predict_proba(passenger.values)[0, 1])
p_trees = np.array(p_trees)

# Mean of the trees compared with the forest
print('First 10 trees:', p_trees[:10].round(2))
print('Mean over all 500 trees:', round(p_trees.mean(), 3))
print('rf.predict_proba:', round(rf.predict_proba(passenger)[0, 1], 3))
print('Share of trees voting "survived":', round(np.mean(p_trees >= 0.5), 3))

**Solution:**

a) OOB accuracy ≈ 0.83, test accuracy ≈ 0.80 – the OOB estimate is a good substitute for a validation set (but not
for the final test set).

b) The passenger is a 24-year-old man in 3rd class. The single trees give survival probabilities between 0 and about
0.23 (the class shares in their leaves). The mean over all 500 trees (≈ 0.13) is exactly what `rf.predict_proba()`
returns: scikit-learn averages the probabilities. A majority vote would give an even lower "probability" (only ≈ 6 % of
the trees vote "survived").

## Exercise 6: «Putting it together in scikit-learn»

a) Run the code of section «Putting it together in scikit-learn» of the concept notebook on the Titanic data: a `GridSearchCV` for a
`RandomForestClassifier(random_state=42)` with `{'max_features': ['sqrt', 0.5], 'min_samples_leaf': [1, 5, 20]}`,
`cv=5`, `scoring='roc_auc'`; then the test AUC and the confusion matrix for the threshold 0.5.

b) Do the same for a `GradientBoostingClassifier(random_state=42)` with
`{'learning_rate': [0.05, 0.1], 'n_estimators': [100, 300], 'max_depth': [2, 3]}`.

c) Which hyperparameters were selected? Interpret.

In [ ]:
# a) Random forest as in the concept notebook
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {'max_features': ['sqrt', 0.5], 'min_samples_leaf': [1, 5, 20]},
    cv=5,
    scoring='roc_auc',
)
grid.fit(X_train, y_train)
p = grid.predict_proba(X_test)[:, 1]
print('Best parameters:', grid.best_params_)
print('CV AUC:', round(grid.best_score_, 3))
print('Test AUC:', round(roc_auc_score(y_test, p), 3))
print(confusion_matrix(y_test, p >= 0.5))

# b) Gradient boosting
grid_gb = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    {
        'learning_rate': [0.05, 0.1],
        'n_estimators': [100, 300],
        'max_depth': [2, 3],
    },
    cv=5,
    scoring='roc_auc',
)
grid_gb.fit(X_train, y_train)
p_gb = grid_gb.predict_proba(X_test)[:, 1]
print('Best parameters:', grid_gb.best_params_)
print('CV AUC:', round(grid_gb.best_score_, 3))
print('Test AUC:', round(roc_auc_score(y_test, p_gb), 3))
print(confusion_matrix(y_test, p_gb >= 0.5))

**Solution:**

a) Random forest: best parameters `max_features='sqrt'`, `min_samples_leaf=5` (CV AUC ≈ 0.88), test AUC ≈ 0.84.
Confusion matrix (rows: actual died/survived, columns: predicted): 97 / 13 and 23 / 46 → 46 of 69 survivors are
recognised (recall ≈ 0.67).

b) Gradient boosting: best parameters `learning_rate=0.05`, `n_estimators=100`, `max_depth=2` (CV AUC ≈ 0.88), test AUC
≈ 0.83; it recognises fewer survivors at the threshold 0.5 (39 of 69) but makes fewer false alarms.

c) Both searches prefer **regularised** models: leaves with at least 5 passengers for the forest, few and very small
trees with a small learning rate for boosting. With only 712 training passengers, more complex models overfit.

## Exercise 7: Model comparison and interpretation

a) Plot the ROC curves of the depth-3 tree, the pruned tree, the tuned random forest and the tuned gradient boosting on
the test set and print the AUCs.

b) Compute the **permutation importance** of the tuned random forest on the test set (`scoring='roc_auc'`, 20 repeats)
and plot it.

c) What do the results tell us about the evacuation of the Titanic?

In [ ]:
# Exercise 7: the four models
models = {
    'Tree (depth 3)': tree_3,
    'Pruned tree': pruned_tree,
    'Random forest': grid.best_estimator_,
    'Gradient boosting': grid_gb.best_estimator_,
}

# a) ROC curve and test AUC of each model
for name, model in models.items():
    p = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, label=name)
    print(name, '- test AUC:', round(roc_auc_score(y_test, p), 3))

# Figure showing the ROC curves
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate (recall)')
plt.title('ROC curves on the test set')
plt.legend()
plt.show()

# b) Permutation importance of the random forest (decrease in test AUC)
perm = permutation_importance(
    grid.best_estimator_,
    X_test,
    y_test,
    scoring='roc_auc',
    n_repeats=20,
    random_state=42,
    n_jobs=-1,
)
importance = pd.Series(perm.importances_mean, index=X_test.columns)
importance = importance.sort_values()
print(importance.round(3))

# Figure showing the permutation importance
importance.plot(kind='barh')
plt.xlabel('Decrease in test AUC when the feature is shuffled')
plt.title('Permutation importance (random forest)')
plt.show()

**Solution:**

a) Test AUCs: random forest ≈ 0.835, gradient boosting ≈ 0.833, depth-3 tree ≈ 0.818, pruned tree ≈ 0.782. The
ensembles are best, but the gain over a simple tree is moderate – the Titanic data contain few features and the main
pattern (sex, class, age) can already be captured by a few rules.

b) **Sex** is by far the most important feature (shuffling it reduces the AUC by about 0.2), followed by class, fare and
age; the number of relatives and the port of embarkation hardly matter.

c) The results reflect the rule "women and children first" and the social order on board: women and children were
preferably assigned to the lifeboats, and passengers in the upper classes (cabins on the upper decks, closer to the
lifeboats) had much better chances than those in 3rd class. Remember that importance describes what the model uses
for prediction – the interpretation as a cause comes from historical knowledge, not from the model.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')